# 02 · Forecast backtest

Delivery day D is forecast at 12:00 on D-1, monthly refit on an expanding window.
Model selection on the validation year (2024-11 – 2025-10), final evaluation once on the test year (2025-11 – 2026-10).

In [1]:
from datetime import date

import polars as pl
import plotly.io as pio

from energy_forecast import features, forecast, plots
from energy_forecast.data import ROOT, load
from energy_forecast.features import BASE_FEATURES
from energy_forecast.forecast import PARAMS, TEST, VAL, Config

pio.templates.default = "plotly_white"
df = features.build(load())

## Model selection on the validation year

In [2]:
CONFIGS = [
    Config("base", features=tuple(BASE_FEATURES), anchor=None, window_days=None),
    Config("anchor", features=tuple(BASE_FEATURES), window_days=None),
    Config("anchor_feat", window_days=None),
    Config("anchor_feat_3y", window_days=3 * 365),
    Config("anchor_feat_2y", window_days=2 * 365),
    Config("anchor_feat_l2", window_days=None, params={**PARAMS, "objective": "l2"}),
]

In [3]:
%%time
val = pl.concat(
    [forecast.seasonal_naive(df, 24, VAL), *(forecast.backtest(df, cfg, VAL) for cfg in CONFIGS)]
)
forecast.score(df, val)

CPU times: user 9min 55s, sys: 11min 1s, total: 20min 56s
Wall time: 13min 3s


model,mae,rmse,bias,hours
str,f64,f64,f64,u32
"""anchor_feat_2y""",13.684983,27.991112,-1.744418,8760
"""anchor_feat_3y""",14.182045,28.345085,-1.024392,8760
"""anchor_feat""",14.476606,28.638502,-0.824371,8760
"""anchor_feat_l2""",14.786911,28.146317,-0.53623,8760
"""base""",17.082953,30.494804,0.304066,8760
"""anchor""",17.52753,31.219449,-0.029043,8760
"""naive_24h""",28.696451,47.079803,0.034369,8760


In [4]:
plots.mae_by(df, val, "month")

## Test year

`forecast.DEFAULT` holds the selected configuration.

In [5]:
%%time
test = pl.concat(
    [
        forecast.seasonal_naive(df, 24, TEST),
        forecast.seasonal_naive(df, 168, TEST),
        forecast.backtest(df, forecast.DEFAULT, TEST),
    ]
)
forecast.score(df, test)

CPU times: user 2min 43s, sys: 1min 14s, total: 3min 57s
Wall time: 2min 45s


model,mae,rmse,bias,hours
str,f64,f64,f64,u32
"""lightgbm""",15.047489,25.539499,-3.009362,8760
"""naive_24h""",31.261788,48.844408,-0.433352,8760
"""naive_168h""",40.200137,61.606403,-2.646073,8760


In [6]:
plots.mae_by(df, test, "month")

In [7]:
plots.mae_by(df, test, "hour")

In [8]:
plots.forecast_lines(df, test, date(2026, 6, 15), date(2026, 6, 21))

In [9]:
plots.forecast_lines(df, test, date(2026, 10, 12), date(2026, 10, 18))

## Save predictions

In [10]:
out = ROOT / "data/processed"
out.mkdir(parents=True, exist_ok=True)
test.write_parquet(out / "backtest.parquet")